## **Exploratory Data Analysis**

#### **Dataset**

We decided to go on the adversarial QA dataset as it a very interesting one to evaluate the flan-T5 model, how it performs initially on an adversial dataset without fine-tuning and on different quantisation level, to see how quantisation affect performances. Then later on we will fine tune the model on the dataset in order to see how fine tuning affect the performance of the model on the different quantisation levels. 

In [18]:
import torch
import numpy as np
import pandas as pd
from datasets import load_dataset
from transformers import AutoTokenizer
device = "cuda:0" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"

In [19]:
model_id = "google/flan-t5-base"
ds = load_dataset("UCLNLP/adversarial_qa", "adversarialQA")

Let us explore the data set

In [14]:
ds

DatasetDict({
    train: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers', 'metadata'],
        num_rows: 30000
    })
    validation: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers', 'metadata'],
        num_rows: 3000
    })
    test: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers', 'metadata'],
        num_rows: 3000
    })
})

In [20]:
from collections import Counter
print(Counter(m["model_in_the_loop"] for m in ds["train"]["metadata"]))

Counter({'Combined': 30000})


In [16]:
for dset in ds:
    n = len(ds[dset])
    n_empty = sum(1 for a in ds[dset]["answers"] if len(a["text"]) == 0)
    print(f"{dset:12s} {n:6d} rows | {n_empty:5d} rows without answer")

train         30000 rows |     0 rows without answer
validation     3000 rows |     0 rows without answer
test           3000 rows |  3000 rows without answer


As we can see the dataset is already split into train, validation and test sets. As mentioned in the documentation, https://huggingface.co/datasets/UCLNLP/adversarial_qa, there are no answer provided with the test set. We'll therefore ignore it, use the current validation set as our test set, and modify the train set to include 27k examples instead of 30k, keeping the 3k for a new validation set.

In [17]:
split = ds["train"].train_test_split(test_size=3000, seed=42)
train_ds = split["train"]
val_ds = split["test"]
test_ds = ds["validation"]
lengths = {"train":len(train_ds), "val":len(val_ds), "test":len(test_ds)}

ds = {"train":train_ds, "val":val_ds, "test":test_ds}

for name, dset in ds.items():
    n = len(dset)
    n_empty = sum(1 for answer in dset["answers"] if len(answer["text"]) == 0)
    print(f"{name:12s} {n:6d} rows | {n_empty:5d} rows without answer")

train         27000 rows |     0 rows without answer
val            3000 rows |     0 rows without answer
test           3000 rows |     0 rows without answer


Let us explore also the tokens length of the context + question that we will later use for the prompt of the model. We'll use for that the tokenizer of the model we chose for the project, which is flan-t5.

In [7]:
tokenizer = AutoTokenizer.from_pretrained(model_id)

In [21]:
for name_dset, dset in ds.items():
    prompts = [f"question: {q}  context: {c}" for q, c in zip(dset["question"], dset["context"])]
    targets = [target[0] for target in dset["answers"]["text"]]
    prompts_len = [len(tokenizer(prompt).input_ids) for prompt in prompts]
    targets_len = [len(tokenizer(target).input_ids) for target in targets]

    def describe_p(name_dataset: str, length: list):
        length = np.array(length)
        print(f"Full prompt (question + context) - {name_dataset}:" )
        print(f"median: {np.median(length):.0f} tokens" )
        print(f"mean: {np.mean(length):.0f} tokens" )
        print(f"max: {np.max(length)} tokens" )
        print(f"min: {np.min(length)} tokens" )
        print(f"share > 512tokens: {(length > 512).mean():.2%}")
        print(f"p95: {np.percentile(length,95):.0f} tokens\n" )

    def describe_t(name_dataset: str, length: list):
        length = np.array(length)
        print(f"Answer - {name_dataset}:" )
        print(f"median: {np.median(length):.0f} tokens" )
        print(f"mean: {np.mean(length):.0f} tokens" )
        print(f"max: {np.max(length)} tokens" )
        print(f"min: {np.min(length)} tokens" )
        print(f"share > 512tokens: {(length > 512).mean():.2%}")
        print(f"p95: {np.percentile(length,95):.0f} tokens\n" )

    describe_p(name_dset, prompts_len)
    describe_t(name_dset, targets_len)

Full prompt (question + context) - train:
median: 175 tokens
mean: 189 tokens
max: 843 tokens
min: 40 tokens
share > 512tokens: 0.41%
p95: 333 tokens

Answer - train:
median: 4 tokens
mean: 7 tokens
max: 386 tokens
min: 2 tokens
share > 512tokens: 0.00%
p95: 22 tokens

Full prompt (question + context) - validation:
median: 178 tokens
mean: 187 tokens
max: 501 tokens
min: 46 tokens
share > 512tokens: 0.00%
p95: 317 tokens

Answer - validation:
median: 4 tokens
mean: 6 tokens
max: 81 tokens
min: 2 tokens
share > 512tokens: 0.00%
p95: 15 tokens



IndexError: list index out of range

In [233]:
for name, dset in ds.items():
    print(name)

train
val
test


In [192]:
x = [x for x in val_ds["answers"]["text"]]
x = [x for x in ds["train"]["answers"]["text"]]